In [ ]:
language = 'pt'

In [ ]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

GRAVAR_AUDIO_JS = """
const pausar = tempo => new Promise(resolve => setTimeout(resolve, tempo));
const blob_para_base64 = blob => new Promise(resolve => {
  const leitorArquivo = new FileReader();
  leitorArquivo.onloadend = () => resolve(leitorArquivo.result);
  leitorArquivo.readAsDataURL(blob)
})
var gravar = tempo => new Promise(async resolve => {
  fluxoAudio = await navigator.mediaDevices.getUserMedia({audio: true})
  gravador = new MediaRecorder(fluxoAudio)
  partesAudio = []
  gravador.ondataavailable = e => partesAudio.push(e.data)
  gravador.start()
  await pausar(tempo)
  gravador.onstop = async () => {
    audioBlob = new Blob(partesAudio)
    audioBase64 = await blob_para_base64(audioBlob)
    resolve(audioBase64)
  }
  gravador.stop()
})
"""

def gravar_audio(segundos=5):
  display(Javascript(GRAVAR_AUDIO_JS))
  resultado_js = output.eval_js('gravar(%s)' % (segundos * 1000))
  audio = b64decode(resultado_js.split(',')[1])

  nome_arquivo = 'audio_gravado.wav'
  with open(nome_arquivo, 'wb') as f:
    f.write(audio)

  return f'/content/{nome_arquivo}'

print('Pode falar...\n')
nome_arquivo = gravar_audio()

display(Audio(nome_arquivo, autoplay=True))



Pode falar...



<IPython.core.display.Javascript object>

In [ ]:
!pip install git+https://github.com/openai/whisper.git
!sudo apt update && sudo apt install ffmpeg


  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-hm3q90g6
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-hm3q90g6
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/pp

In [ ]:
import whisper

modelo = whisper.load_model("small")

In [ ]:
resultado = modelo.transcribe("audio_gravado.wav", language="pt")
print("Você disse:", resultado["text"])

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Você disse:  Agenda!


In [ ]:
import webbrowser

from IPython.display import HTML

display(HTML('<a href="https://calendar.google.com/calendar/u/0/r" target="_blank">Clique aqui para abrir a Agenda Google</a>'))

comando = resultado["text"].lower()

if "abrir agenda" in comando or "agenda" in comando:
  print("Abrindo sua Agenda Google...")
  display(HTML('<a href="https://calendar.google.com/calendar/u/0/r" target="_blank">Clique aqui para abrir a Agenda Google</a>'))
elif "abrir youtube" in comando or "youtube" in comando:
  print("Abrindo Youtube...", )
  display(HTML('<a href="https://youtube.com/" target="_blank">Abrir YouTube</a>'))
else:
  print("Desculpe, não entendi o comando:", comando)

Abrindo sua Agenda Google...


In [ ]:
!pip install gTTS

In [ ]:
from gtts import gTTS
from IPython.display import Audio

# Suponha que o Whisper retornou o texto transcrito
comando = resultado["text"].lower()

if "agenda" in comando:
    frase = "Abrindo sua Agenda Google..."
    nome_arquivo = "abrindo_agenda.mp3"
elif "youtube" in comando:
    frase = "Abrindo YouTube..."
    nome_arquivo = "abrindo_youtube.mp3"
else:
    frase = "Desculpe, não entendi o comando."
    nome_arquivo = "comando_nao_reconhecido.mp3"

# Gera o áudio com gTTS
tts = gTTS(text=frase, lang="pt")
tts.save(nome_arquivo)

# Exibe o player de áudio no Colab
display(Audio(nome_arquivo, autoplay=True))


# Assistente de Voz Simples com Whisper e gTTS

Este projeto demonstra um assistente de voz básico construído no Google Colab, utilizando as bibliotecas Whisper (para reconhecimento de fala) e gTTS (para síntese de fala). O assistente pode gravar áudio, transcrever comandos e responder com feedback de áudio, além de realizar ações simples como abrir a Agenda Google ou o YouTube.

## Funcionalidades

*   **Gravação de Áudio**: Captura áudio diretamente do microfone do usuário no navegador.
*   **Reconhecimento de Fala (Speech-to-Text)**: Transcreve o áudio gravado para texto usando o modelo `small` da OpenAI Whisper, com suporte para português.
*   **Reconhecimento de Comandos**: Analisa o texto transcrito para identificar comandos específicos como "agenda" ou "youtube".
*   **Síntese de Fala (Text-to-Speech)**: Gera respostas de áudio usando a biblioteca gTTS, fornecendo feedback verbal ao usuário.
*   **Ações Simples**: Abre URLs específicas no navegador com base nos comandos reconhecidos (ex: Google Calendar, YouTube).

## Como Funciona

O notebook é dividido em etapas lógicas:

1.  **Definição do Idioma**: Define o idioma principal para o reconhecimento e síntese de fala como português (`pt`).
2.  **Gravação de Áudio**: Um trecho de código JavaScript permite gravar o microfone do usuário por um tempo determinado e salva o áudio como `audio_gravado.wav`.
3.  **Instalação de Dependências**: Instala o Whisper e o FFmpeg, que são essenciais para o processamento de áudio e reconhecimento de fala.
4.  **Carregamento do Modelo Whisper**: Um modelo pré-treinado do Whisper (`small`) é carregado para realizar a transcrição.
5.  **Transcrição e Reconhecimento de Comandos**: O áudio gravado é transcrito, e o texto resultante é analisado para identificar comandos chave.
6.  **Resposta e Ação**: Baseado no comando, o assistente pode:
    *   Imprimir uma mensagem na tela.
    *   Gerar um link clicável para abrir serviços como Google Calendar ou YouTube.
    *   Sintetizar uma resposta de áudio usando gTTS e reproduzi-la.

## Instalação e Uso no Google Colab

Para executar este projeto no Google Colab:

1.  **Abra o Notebook**: Faça o upload ou clone este notebook para o seu ambiente Colab.
2.  **Execute as Células Sequencialmente**:
    *   Execute a célula que define `language = 'pt'`.
    *   Execute a célula que contém a função `gravar_audio()` e as chamadas para gravação. Você será solicitado a permitir o acesso ao seu microfone.
    *   Execute as células que instalam o Whisper e o FFmpeg (`!pip install git+...` e `!sudo apt update && sudo apt install ffmpeg`).
    *   Execute a célula que importa o Whisper e carrega o modelo (`modelo = whisper.load_model('small')`).
    *   Execute a célula de transcrição (`resultado = modelo.transcribe(...)`) para converter seu áudio em texto.
    *   Execute a célula que contém a lógica de reconhecimento de comandos e exibição de links (`import webbrowser...`).
    *   Execute as células que instalam o `gTTS` (`!pip install gTTS`) e geram e reproduzem a resposta de áudio.

## Dependências Principais

*   `openai-whisper`: Para reconhecimento de fala.
*   `gTTS`: Para síntese de fala.
*   `ffmpeg`: Requisito para o Whisper processar arquivos de áudio.
*   `IPython.display`: Para exibir áudio e JavaScript no Colab.

## Exemplo de Interação

Quando a célula de gravação for executada, você ouvirá "Pode falar..." e poderá dizer um comando como "Agenda" ou "YouTube". O assistente irá processar seu comando e responder de acordo.